<a href="https://colab.research.google.com/github/usharani-192312304/ITA0605-ML/blob/main/Exp12_Iris_KNN.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Experiment 12: Iris Flower Classification using K-NN
**Course:** ITA06 - Machine Learning (Lab)

## Aim
Implement Iris Flower Classification using K-Nearest Neighbours.

## Background
The **Iris** dataset (Fisher, 1936) has 150 flowers from 3 species (*setosa, versicolor, virginica*), each described by sepal length/width and petal length/width. K-NN classifies a flower by majority vote among its `k` nearest neighbours in feature space; distances are meaningful only after **scaling** the features.

## Pipeline followed in this notebook
1. Step 1: Data Collection
2. Step 2: Data Loading
3. Step 3: Data Preprocessing
4. Step 4: Exploratory Data Analysis (EDA)
5. Step 5: Model Training
6. Step 6: Model Evaluation
7. Step 7: Prediction on New Data

> **How to run:** Open in Google Colab -> *Runtime -> Run all*. No files need to be uploaded; the data is created or loaded inside the notebook.

## Setup: Import libraries

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings

warnings.filterwarnings("ignore")
sns.set_style("whitegrid")
plt.rcParams["figure.figsize"] = (8, 5)
np.random.seed(42)
print("Libraries imported successfully")

## Step 1: Data Collection
Load the Iris dataset bundled with scikit-learn and store it as a CSV.

In [ ]:
from sklearn.datasets import load_iris

iris = load_iris(as_frame=True)
frame = iris.frame.copy()
frame["species"] = frame["target"].map(dict(enumerate(iris.target_names)))
frame = frame.drop(columns="target")
frame.columns = ["sepal_length", "sepal_width", "petal_length", "petal_width", "species"]
frame.to_csv("iris.csv", index=False)
print("Saved iris.csv", frame.shape)

## Step 2: Data Loading

In [ ]:
df = pd.read_csv("iris.csv")
df.head()

## Step 3: Data Preprocessing
Encode species as numbers, split (stratified) and standardise features using the training set statistics.

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder

print("Missing values:", df.isnull().sum().sum(), "| Duplicate rows:", df.duplicated().sum())
df = df.drop_duplicates().reset_index(drop=True)

features = ["sepal_length", "sepal_width", "petal_length", "petal_width"]
le = LabelEncoder()
y = le.fit_transform(df["species"])
X = df[features]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)
scaler = StandardScaler().fit(X_train)
X_train_s, X_test_s = scaler.transform(X_train), scaler.transform(X_test)
print("Classes:", list(le.classes_), "| Train:", X_train_s.shape, "Test:", X_test_s.shape)

## Step 4: Exploratory Data Analysis (EDA)

In [ ]:
print(df.describe().T.round(2))
print("\n", df["species"].value_counts())

In [ ]:
sns.pairplot(df, hue="species", corner=True, height=2.2)
plt.suptitle("Pairwise relationships in the Iris data", y=1.02); plt.show()

fig, axes = plt.subplots(1, 4, figsize=(18, 4))
for ax, col in zip(axes, features):
    sns.boxplot(data=df, x="species", y=col, ax=ax)
plt.tight_layout(); plt.show()

sns.heatmap(df[features].corr(), annot=True, cmap="coolwarm", center=0); plt.title("Correlation"); plt.show()

## Step 5: Model Training
Tune `k`, the weighting scheme and the distance metric using **grid search with 5-fold cross-validation**.

In [ ]:
from sklearn.neighbors import KNeighborsClassifier
from sklearn.model_selection import GridSearchCV, cross_val_score

param_grid = {"n_neighbors": list(range(1, 21)), "weights": ["uniform", "distance"], "metric": ["euclidean", "manhattan"]}
grid = GridSearchCV(KNeighborsClassifier(), param_grid, cv=5, scoring="accuracy").fit(X_train_s, y_train)
print("Best parameters:", grid.best_params_)
print("Best CV accuracy:", round(grid.best_score_, 4))

ks = range(1, 31)
cv_acc = [cross_val_score(KNeighborsClassifier(n_neighbors=k), X_train_s, y_train, cv=5).mean() for k in ks]
plt.plot(ks, cv_acc, marker="o"); plt.xlabel("k"); plt.ylabel("5-fold CV accuracy"); plt.title("Effect of k (default settings)"); plt.show()

knn = grid.best_estimator_

## Step 6: Model Evaluation
Test-set accuracy, classification report, confusion matrix and decision regions in the petal-length / petal-width plane.

In [ ]:
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

y_pred = knn.predict(X_test_s)
print("Test accuracy:", round(accuracy_score(y_test, y_pred), 4))
print(classification_report(y_test, y_pred, target_names=le.classes_))
sns.heatmap(confusion_matrix(y_test, y_pred), annot=True, fmt="d", cmap="Blues", xticklabels=le.classes_, yticklabels=le.classes_)
plt.xlabel("Predicted"); plt.ylabel("Actual"); plt.title("K-NN on Iris - Confusion matrix"); plt.show()

In [ ]:
# Decision regions using only the two petal features
cols = [2, 3]
knn2 = KNeighborsClassifier(**grid.best_params_).fit(X_train_s[:, cols], y_train)
xx, yy = np.meshgrid(np.linspace(X_train_s[:, 2].min() - .5, X_train_s[:, 2].max() + .5, 300),
                     np.linspace(X_train_s[:, 3].min() - .5, X_train_s[:, 3].max() + .5, 300))
Z = knn2.predict(np.c_[xx.ravel(), yy.ravel()]).reshape(xx.shape)
plt.contourf(xx, yy, Z, alpha=0.3, cmap="viridis")
plt.scatter(X_train_s[:, 2], X_train_s[:, 3], c=y_train, cmap="viridis", edgecolor="k")
plt.xlabel("petal length (scaled)"); plt.ylabel("petal width (scaled)"); plt.title("K-NN decision regions"); plt.show()

## Step 7: Prediction on New Data

In [ ]:
new_flowers = pd.DataFrame([[5.1, 3.5, 1.4, 0.2], [6.0, 2.9, 4.5, 1.5], [6.9, 3.1, 5.8, 2.1]], columns=features)
pred = knn.predict(scaler.transform(new_flowers))
proba = knn.predict_proba(scaler.transform(new_flowers))
for row, p, pr in zip(new_flowers.values, pred, proba):
    print(row, "->", le.classes_[p], dict(zip(le.classes_, pr.round(2))))

## Conclusion
K-NN classifies Iris with about 93-100% test accuracy. *Setosa* is perfectly separable; the only confusion occurs between *versicolor* and *virginica*, whose petals overlap slightly. Petal measurements are the most discriminative features. Because the test set is just 30 flowers, the cross-validated accuracy (~95%) is a more reliable estimate than a single split.